In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import os
from tqdm import tqdm

# Convert VCF files to a big mutations frame

This will process the main experiment as well as the satellite mini-experiments:
* IA to IH: initial replay of the first 2,500 generations
* AA to AH: when lineage A cross-contaminated all the other lineages around 30,000 generations, and I only realized running the experiment for ~10,000 more generations. I then restarted from the 30,000 generations checkpoint, but I also sequenced the 8 evolved offshoots of population A. It has interesting stuff, actually.

The main experiment's populations names will simply be "A" to "H".
In R, you can select only the main experiment by filtering according to `nchar(pop) == 1` (i.e., population name is one-character-long).


### Identify the mutations
* SNP, SUB, DEL, INS
* Base number: 0 for a SNP, -1 for a 1bp deletion, +1 for a 1bp insertion etc. To calculate frameshifts and stretch extensions
* Stretch length: original (WT) number of bp in the stretch where the mutation appears. That is the base corresponding to position
* For stretch insertion/deletion, position is the first base of the stretch

In [2]:
# Read reference genome
genome = dict()

# E. coli
with open('../../../datasets/genomics/reference_genome/U00096_raw.txt') as f:
    #next(f)
    genome['U00096'] = f.read().replace('\n', '')

# Phage Phi80 = Lula
with open('../../../datasets/genomics/reference_genome/JX871397_raw.txt') as f:
    #next(f)
    genome['JX871397'] = f.read().replace('\n', '')

In [3]:
def trim(ref, alt):
    '''If ALT matches the beginning of REF (or the opposite), trim both.'''
    if ref.startswith(alt):
        return ref[len(alt):], ''
    elif alt.startswith(ref):
        return '', alt[len(ref):]
    else:
        return ref, alt

def abbreviate(seq, size = 5):
    if len(seq) > pad * 2:
        return f'{seq[:size]}...{seq[-size:]} ({len(seq)} bp)'
    else:
        return seq
    
def find_repeat_start(chrom, pos):
    # For repeat number change mutations, find where the repeat starts
    #pos -= 1 # the repeat's base is the one before pos
    letter = genome[chrom][pos]
    
    while genome[chrom][pos - 1] == letter: # check the base before
        pos -= 1
        
    return pos

def find_repeat_end(chrom, pos):
    # Find the length of a repeat on the genome, starting at pos
    letter = genome[chrom][pos]
    p = pos + 1
    
    while genome[chrom][p] == letter:
        p += 1
        
    return p

In [4]:
folder = '../../../datasets/genomics/vcf/'
#vcf_files = [i for i in sorted(os.listdir(folder)) if i[0] != '.' and 'EF' in i]
vcf_files = [i for i in sorted(os.listdir(folder)) if i[0] != '.']
', '.join(vcf_files)

rows_to_skip = 9 # 7 + number of reference sequences

## Parse VCF files and give mutations consistent positions

In [5]:
pad = 8
mutation_dfs = []

out_of_bond_dfs = []

#for file in ['EF13.vcf', 'EF14.vcf']:
for file in tqdm(vcf_files):
    #print(file)

    vcf = pd.read_csv(folder + file, skiprows = rows_to_skip, sep = '\t').drop(columns = ['ID', 'FILTER'])
        
    ##############################################
    ### Add sample, frequency and quality info ###
    ##############################################

    ### Add sample info
    if file[0] == 'E': # main line of experiment
        vcf['name'] = file[0:4]
        vcf['pop'] = file[1]
        vcf['gen'] = int(file[2:4])
        
    elif file[0] == 'A': # A takeover (label as gen -2)
        vcf['name'] = file[0:2]
        vcf['pop'] = file[0:2]
        vcf['gen'] = -2
        
    elif file[0] == 'I': # Initial replay (label as gen -1)
        vcf['name'] = file[0:2]
        vcf['pop'] = file[0:2]
        vcf['gen'] = -1
    
    # On rare occasions INFO only contains the allele frequency. Set read counts to zero.
    vcf.loc[(vcf.INFO.str.count(';') != 2), 'INFO'] += ';AD=0;DP=0'   
    
    # This also happens sometimes:
    if len(vcf[vcf.ALT.astype(str) == 'nan']):
        #display(vcf[vcf.ALT.astype(str) == 'nan'])
        vcf.loc[vcf.ALT.astype(str) == 'nan', 'ALT'] = ''

    # INFO is Allele Frequency; Allele Depth; Total Depth
    vcf['freq'] = vcf.apply(lambda x: x.INFO.replace('=', ';').split(';')[1], axis = 1).apply(pd.to_numeric)
    vcf['allele_depth'] = vcf.apply(lambda x: x.INFO.replace('=', ';').split(';')[3], axis = 1).apply(pd.to_numeric)
    vcf['total_depth'] = vcf.apply(lambda x: x.INFO.replace('=', ';').split(';')[5], axis = 1).apply(pd.to_numeric)
    vcf['QUAL'] = vcf.apply(lambda x: 0 if x.QUAL == '.' else float(x.QUAL), axis = 1)
    
    vcf.rename(columns={'#CHROM':'chrom'}, inplace=True)
    
    ##########################################
    ### Simplify the ref and alt sequences ###
    ##########################################
    
    # INITIAL STATE FROM BRESEQ
    # POS is 1-indexed
    # POS - 1 is always the first base of REF
    # For SNPs, POS - 1 is the position of the SNP
    # For everything else, one base is added at the beginning, so the real start of the mutation is POS
      
    # Convert POS to zero-indexed
    vcf['POS'] = vcf.POS - 1
    
    # If REF and ALT include each other, cut the first bases off (usual for INS and DEL) 
    vcf[['ref', 'alt']] = vcf.apply(lambda x: trim(x.REF, x.ALT), axis = 1, result_type = 'expand')
    
    # Rename the native columns to keep as archive
    vcf['vcf_pos'] = vcf['POS']
    vcf['raw_ref'] = vcf['ref'] # for ref and alt, keep the truncated version
    vcf['raw_alt'] = vcf['alt']
    
    # Detect cases where POS must be moved: SNPs and REPs
    vcf['is_snp'] = (vcf.REF.str.len() == 1) & (vcf.ALT.str.len() == 1)
    
    # Adjust POS to account for the trimming
    vcf['POS'] = vcf.POS + vcf.REF.str.len() - vcf.ref.str.len() # For REPs, POS is now the last letter of the stretch
    
    # Now, for INS, pos is the coordinate of the first base that will be shifted right by the insertion
    
    #################################################
    ### Fix the position of repeat number changes ###
    #################################################
    
    # Warn against positions larger than the chromosome length (wtf)
    out_of_bond = vcf[(vcf.POS > 4641582)]
    if len(out_of_bond):
        out_of_bond_dfs.append(out_of_bond)
        
        vcf = vcf[(vcf.POS <= 4641582)].reset_index(drop = True)
    
    # Find the stretch of nucleotides where the mutation happens
    vcf['base_at_POS'] = vcf.apply(lambda x: genome[x.chrom][x.POS], axis = 1)
    vcf['rep_probe'] = np.where((vcf.ref == '') & (vcf.base_at_POS != vcf.alt), vcf.POS - 1, vcf.POS) # INS, must look at the nucleotide before
    vcf['rep_start'] = vcf.apply(lambda x: find_repeat_start(x.chrom, x.rep_probe), axis = 1)
    vcf['rep_end'] = vcf.apply(lambda x: find_repeat_end(x.chrom, x.rep_start), axis = 1)
    vcf['rep_seq'] = vcf.apply(lambda x: genome[x.chrom][x.rep_start:x.rep_end], axis = 1)
    vcf['rep_len'] = vcf.rep_end - vcf.rep_start

    # For REPs set pos as the beginning of the stretch
    vcf['is_rep'] = vcf.apply(lambda x: len(set(str(x['ref']) + str(x['alt'] + str(x['rep_seq'])))) == 1, axis = 1)
    vcf['rep_nt'] = np.where(vcf.is_rep, vcf.rep_seq.str[:1], '.')
    vcf['pos'] = np.where(vcf.is_rep, vcf.rep_start, vcf.POS)
    
    # Set the end of the modified region
    vcf['pos_end'] = np.where(vcf.is_rep, vcf.rep_end, vcf.pos + vcf.ref.str.len())
    
    # For the repeat number variations, this should match the repeat whose length changes
    vcf['delta'] = vcf.alt.str.len() - vcf.ref.str.len()
    
    # Make abbreviated, prettied-up sequences    
    vcf['ref'] = vcf['ref'].apply(abbreviate)
    vcf['alt'] = vcf['alt'].apply(abbreviate)
        
    # Add local context (ref is surrounded by dots. For REPs, show the whole stretch as the ref)
    vcf.loc[~vcf.is_rep, 'locus'] = vcf.apply(lambda x: genome[x.chrom][x.pos - pad:x.pos] + '.' + x.ref + '.' + genome[x.chrom][x.pos_end:x.pos_end + pad + 1], axis = 1)
    vcf.loc[vcf.is_rep, 'locus'] = vcf.apply(lambda x: genome[x.chrom][x.pos - pad:x.pos] + '.' + x.rep_seq + '.' + genome[x.chrom][x.pos_end:x.pos_end + pad + 1], axis = 1)
    #vcf.loc[~vcf.is_rep, 'raw_locus'] = vcf.apply(lambda x: genome[x.chrom][x.pos - 3:x.pos_end + 3], axis = 1)

    # Attribute mutation type
    vcf['type'] = 'UNK' # Unknown (should all be gone)
    vcf.loc[~(vcf.is_rep) & (vcf.ref.str.len() >= 1) & (vcf.alt.str.len() == 0), 'type'] = 'DEL' # Deletions
    vcf.loc[~(vcf.is_rep) & (vcf.ref.str.len() == 0) & (vcf.alt.str.len() >= 1), 'type'] = 'INS' # Insertions
    vcf.loc[~(vcf.is_rep) & (vcf.ref.str.len() >= 1) & (vcf.alt.str.len() >= 1), 'type'] = 'SUB' # Substitutions
    vcf.loc[(vcf.is_snp), 'type'] = 'SNP' # SNPs
    vcf.loc[(vcf.is_rep), 'type'] = 'REP' # REP copy number change

    # Remove unnecessary columns
    vcf.drop(columns = ['POS', 'REF', 'ALT'], inplace = True)
    vcf.drop(columns = ['rep_probe', 'rep_start', 'rep_end', 'rep_seq', 'base_at_POS'], inplace = True)
    vcf.drop(columns = ['is_rep', 'is_snp'], inplace = True)
    vcf['qual'] = vcf.QUAL
    vcf.drop(columns = ['INFO', 'QUAL'], inplace = True)
   
    mutation_dfs.append(vcf)

raw = pd.concat(mutation_dfs).reset_index(drop = True)
raw.sort_values(['pop', 'gen', 'pos', 'type'], ascending = [True, True, True, False], inplace = True)
raw.reset_index(inplace = True, drop = True)

if out_of_bond_dfs:
    print('Out of bond variants:')
    display(pd.concat(out_of_bond_dfs))

  0%|          | 0/209 [00:00<?, ?it/s]

  1%|▏         | 3/209 [00:00<00:09, 21.60it/s]

  3%|▎         | 6/209 [00:00<00:09, 21.81it/s]

  4%|▍         | 9/209 [00:00<00:09, 21.72it/s]

  6%|▌         | 12/209 [00:00<00:08, 22.02it/s]

  7%|▋         | 15/209 [00:00<00:08, 21.83it/s]

... (70 outputs left out)

 99%|█████████▉| 207/209 [00:11<00:00, 15.99it/s]

100%|██████████| 209/209 [00:11<00:00, 18.06it/s]

Out of bond variants:


,chrom,POS,REF,ALT,QUAL,INFO,name,pop,gen,freq,allele_depth,total_depth,ref,alt,vcf_pos,raw_ref,raw_alt,is_snp
761,U00096,4641614,AA,A,2.6,AF=0.0441;AD=3;DP=68,EA04,A,4,0.0441,3.0,68.0,A,,4641613,A,,False
443,U00096,4641635,C,A,1.7,AF=0.0269;AD=12;DP=287,EA18,A,18,0.0269,12.0,287.0,C,A,4641635,C,A,True
496,U00096,4641632,C,A,5.0,AF=0.0221;AD=10;DP=296,EB19,B,19,0.0221,10.0,296.0,C,A,4641632,C,A,True
497,U00096,4641635,C,A,4.0,AF=0.0284;AD=13;DP=297,EB19,B,19,0.0284,13.0,297.0,C,A,4641635,C,A,True
475,U00096,4641635,C,A,2.7,AF=0.0257;AD=11;DP=269,EC20,C,20,0.0257,11.0,269.0,C,A,4641635,C,A,True
674,U00096,4641635,C,A,2.0,AF=0.0361;AD=14;DP=223,EE19,E,19,0.0361,14.0,223.0,C,A,4641635,C,A,True


# Convert pos to 1-indexed

For consistency with the rest of the analysis, genome positions shoud be 1-indexed. *native_pos* remains 0-indexed.

As a standard, the mutS(G696E) variant's **pos** should be **2859179**.

In [6]:
# Convert to 1-indexed
raw1 = raw.copy()
raw1['pos'] = raw1['pos'] + 1
raw1['pos_end'] = raw1['pos_end'] + 1

# Backup the original values of some variables
raw1['raw_pos'] = raw1['pos']
raw1['raw_freq'] = raw1['freq']
raw1['raw_type'] = raw1['type']

# Save to disk
raw1.to_csv('../../../datasets/genomics/01_variant_calling/intermediate/raw_one_indexed.tsv', sep = '\t', index = False)

raw1[raw1.pos == 2859179].head()

,chrom,name,pop,gen,freq,allele_depth,total_depth,ref,alt,vcf_pos,...,rep_nt,pos,pos_end,delta,locus,type,qual,raw_pos,raw_freq,raw_type
52572,U00096,ED06,D,6,0.3333,23.0,69.0,G,A,2859178,...,.,2859179,2859180,0,TGAGATCG.G.GCGTGGAAC,SNP,66.6,2859179,0.3333,SNP
53461,U00096,ED07,D,7,0.0679,5.0,73.0,G,A,2859178,...,.,2859179,2859180,0,TGAGATCG.G.GCGTGGAAC,SNP,3.1,2859179,0.0679,SNP
55009,U00096,ED09,D,9,0.5228,80.0,153.0,G,A,2859178,...,.,2859179,2859180,0,TGAGATCG.G.GCGTGGAAC,SNP,236.1,2859179,0.5228,SNP
57065,U00096,ED11,D,11,0.4729,35.0,74.0,G,A,2859178,...,.,2859179,2859180,0,TGAGATCG.G.GCGTGGAAC,SNP,111.5,2859179,0.4729,SNP
58126,U00096,ED12,D,12,0.1029,7.0,68.0,G,A,2859178,...,.,2859179,2859180,0,TGAGATCG.G.GCGTGGAAC,SNP,13.5,2859179,0.1029,SNP


# Save an abbreviation index

In [7]:
ref_abbreviations = raw1[['bp' in i and i != '' for i in raw1.ref]][['ref', 'raw_ref']].drop_duplicates().reset_index(drop = True)
alt_abbreviations = raw1[['bp' in i and i != '' for i in raw1.alt]][['alt', 'raw_alt']].drop_duplicates().reset_index(drop = True)
ref_abbreviations.columns = alt_abbreviations.columns = ['short', 'long']

In [8]:
abbreviations = pd.concat([ref_abbreviations, alt_abbreviations]).reset_index(drop = True)
abbreviations.to_csv('../../../datasets/genomics/01_variant_calling/results/abbreviations.tsv', sep = '\t', index = False)